# 03 — Feature Engineering

**Goal:** define a future-maintenance target and create predictors using only information available at prediction time.

> Leakage rule: the current or future maintenance label must never be a predictor. Rolling features are shifted by one row before aggregation. Fit preprocessing parameters on the training period only.

In [ ]:
from pathlib import Path
import os
import sys

import numpy as np
import pandas as pd
import polars as pl

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.data_utils import resolve_data_path, scan_tina, validate_event_columns

DATA_PATH = resolve_data_path()
TIME_COL = os.environ.get('TINA_TIME_COLUMN', 'unix')
MAX_ROWS = int(os.environ.get('TINA_FEATURE_ROWS', '1000000'))
HORIZON_STEPS = int(os.environ.get('TINA_HORIZON_STEPS', '1800'))
ROLLING_WINDOWS = [30, 300, 1800]
MAX_SENSOR_FEATURES = int(os.environ.get('TINA_MAX_SENSOR_FEATURES', '20'))
SAVE_FEATURES = os.environ.get('TINA_SAVE_FEATURES', '0').lower() in {'1', 'true', 'yes'}
RANDOM_SEED = 613

## Load the cleaned period

The default uses source data as a scaffold. For final work, save the documented cleaning logic from notebook 02 in `src/` and call it here.

In [ ]:
lazy = scan_tina(DATA_PATH)
validate_event_columns(lazy)
frame = lazy.head(MAX_ROWS).collect()
if TIME_COL not in frame.columns:
    raise KeyError(f'Set TINA_TIME_COLUMN to the verified timestamp column: {frame.columns}')
frame = frame.sort(TIME_COL)
pdf = frame.to_pandas()
print(f'Working shape: {pdf.shape}')

## Define the target

`maintenance_within_horizon` is 1 when at least one maintenance-labelled row occurs in the next `HORIZON_STEPS` observations, excluding the current row. Convert the step horizon to clock time using the verified sampling interval and document it. Group consecutive maintenance-labelled rows into events for final event-level evaluation.

In [ ]:
maintenance_now = (pdf['m_id'].astype(str).str.lower() != 'none').astype('int8')
future_maintenance = (
    maintenance_now.shift(-1)
    .iloc[::-1]
    .rolling(HORIZON_STEPS, min_periods=1)
    .max()
    .iloc[::-1]
    .fillna(0)
    .astype('int8')
)
pdf['maintenance_within_horizon'] = future_maintenance
print(pdf['maintenance_within_horizon'].value_counts(normalize=True).rename('share'))

## Past-only sensor features

In [ ]:
excluded = {TIME_COL, 'm_id', 'm_subid', 'alarms', 'maintenance_within_horizon'}
sensor_columns = [
    column for column in pdf.select_dtypes(include=np.number).columns
    if column not in excluded
][:MAX_SENSOR_FEATURES]  # expand only after memory/runtime measurement

features = pd.DataFrame(index=pdf.index)
features[TIME_COL] = pdf[TIME_COL]
features['maintenance_within_horizon'] = pdf['maintenance_within_horizon']
features['alarm_now'] = (pdf['alarms'].astype(str).str.lower() != 'none').astype('int8')

for column in sensor_columns:
    past = pdf[column].shift(1)
    features[f'{column}__lag1'] = past
    for window in ROLLING_WINDOWS:
        rolling = past.rolling(window=window, min_periods=max(2, window // 10))
        features[f'{column}__mean_{window}'] = rolling.mean()
        features[f'{column}__std_{window}'] = rolling.std()
        features[f'{column}__min_{window}'] = rolling.min()
        features[f'{column}__max_{window}'] = rolling.max()

features = features.iloc[:-HORIZON_STEPS].copy()
print(f'Feature table shape: {features.shape}')
features.head()

## Leakage audit

Before saving a modelling table, answer each item:

- Are the rows sorted by the verified timestamp? **TODO**
- Do all lags/rolling windows exclude the current and future rows? **TODO**
- Are `m_id` and `m_subid` absent from predictors? **TODO**
- Is `alarm_now` genuinely observable at prediction time? **TODO**
- Were rows during maintenance and the recovery period handled explicitly? **TODO**
- Are imputation/scaling fitted only on training data? **TODO**
- Is the final incomplete horizon removed? **TODO**

## Optional local export (ignored by Git)

In [ ]:
OUTPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'tina_features.parquet'
if SAVE_FEATURES:
    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    features.to_parquet(OUTPUT_PATH, index=False)
    print(f'Saved local modelling table: {OUTPUT_PATH}')
else:
    print(f'Set TINA_SAVE_FEATURES=1 after the leakage audit to save locally: {OUTPUT_PATH}')